# 06b · CircuitKIT cross-check of the refusal circuit (toolkit chaining)

Runs Lexsi Labs' **CircuitKIT** (EAP-IG, node level) on the matched calibration pairs and compares its ranking with the components our own attribution + exact patching found in 06. Uses the tested adapter `safetypulse.adapters.circuitkit` (attach the `safetypulse-pkg` dataset).

| | |
|---|---|
| **Kaggle settings** | Accelerator: GPU T4 · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) · `safetypulse-pkg` · outputs of **03 matched**, **04**, **06 matched** · `sp-models` |
| **Reads** | `pairs.jsonl`, `token_sets.json`, `probe_bundle/` |
| **Writes** (`/kaggle/working/`) | `circuitkit/crosscheck.json` |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

In [1]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

{'torch': '2.11.0+cu130',
 'transformers': '5.20.2',
 'cuda_available': True,
 'on_kaggle': True,
 'smoke': False,
 'peft': '0.21.2',
 'gpus': ['Tesla T4'],
 'capability': [7, 5],
 'vram_gb': 15.6}

In [2]:
import json, subprocess, sys
import torch

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "circuitkit==0.2.0"], check=True)
pkg = next(p for p in sp.find_inputs("cli.py") if p.parent.name == "safetypulse")
sys.path.insert(0, str(pkg.parent.parent))
from safetypulse import ProbeBundle
from safetypulse.adapters import circuitkit as ck

MODEL = os.environ.get("SP_MODEL", sp.DEFAULT_MODEL)
N_PAIRS = int(os.environ.get("SP_N_PAIRS", 64)) if not sp.SMOKE else 4   # EAP-IG is memory-heavy on a T4
K = 20
OUT = sp.out_dir("circuitkit")
pairs = [p for p in sp.load_jsonl(sp.find_input("pairs.jsonl")) if p["split"] == "calib"][:N_PAIRS]
token_sets = sp.load_json(sp.find_input("token_sets.json"))[MODEL]
bundle = ProbeBundle.load(sp.find_input("probe_bundle.json").parent)
print(len(pairs), "calib pairs;", len(bundle.components), "components in our bundle; layer", bundle.layer)

StopIteration: 

In [ ]:
with sp.gpu_timer() as t:
    result = ck.crosscheck(sp.model_source(MODEL), pairs, token_sets, bundle, k=K, dtype="float16", batch_size=2)
result["gpu_seconds"] = t["seconds"]
sp.save_json(result, OUT / "crosscheck.json")
{k: v for k, v in result.items() if not isinstance(v, (list, dict))}